# Entrenament model offline - Predicció glucosa

#### Import de les llibreries necessaries

In [80]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error


import warnings

warnings.simplefilter("ignore", FutureWarning)

pd.set_option('display.max_columns', None)

#### Definició de les columnes dels datasets

In [81]:
cols = [
    'year', 'month', 'day', 'hour', 'minute', 'second', # A–F
    'glucose_level', # G
    'finger_stick', # H
    'basal', # I
    'bolus', # J
    'sleep', # K
    'work', # L
    'stressors', # M
    'hypo_event', # N
    'illness', # O
    'exercise', # P
    'basis_heart_rate', # Q
    'basis_gsr', # R
    'basis_skin_temperature', # S
    'basis_air_temperature',  # T
    'basis_step', # U
    'basis_sleep', # V
    'meal', # W
    'meal_type' # X
]

#### Definició de pacients i horitzons

In [82]:
# Definicio de id dels pacients
PACIENTS = [559, 563, 570, 575, 588, 591]

# Definició de l'horitzo
HORITZO = {30:6, 60:12}  # minuts : pas/files (de 5 mins)

## Carrega del dataset

In [83]:
# Funcio per obtenir els datasets dels pacients
def load_data(pacient, train_or_test):
    df=pd.read_csv(f'../data/{pacient}/{pacient}_{train_or_test}.csv', sep=';', header = None, names = cols)
    return df

prova_559_train = load_data(559, 'train')
prova_559_test = load_data(559, 'test')

## Preprocessament del dataset

In [84]:
def preprocess(df):
    prep = df.copy()

    prep['time'] = pd.to_datetime(
        dict(year=df.year, month=df.month, day=df.day, hour=df.hour, minute=df.minute)
    )
    
    prep.sort_values('time', inplace=True)

    # Coma decimal a punt
    convert = ["basal","bolus","basis_gsr","basis_skin_temperature","basis_air_temperature"]
    
    for c in convert:
        prep[c] = (prep[c].astype(str)
                   .str.replace(",",".", regex=False)
                   .str.strip()
                   .astype(float))

    # Unifiquem tipo
    cat_meal = {
        1:"Desayuno",
        2:"Almuerzo",
        3:"Cena",
        4:"Snack",
        5:"Correccion_hipo"
    }

    prep["meal_type"] = prep["meal_type"].map(cat_meal).astype("category")
    prep = pd.get_dummies(prep, columns=['meal_type'], dummy_na=False, prefix='meal')

    # Drop columnas amb casi tot NaN o valor constant
    prep = prep.drop(columns=["second","finger_stick","meal"])

    # Zeros que no poden ser valids
    invalid_zero = [
        "glucose_level",
        "basis_heart_rate",
        "basis_gsr",
        "basis_skin_temperature",
        "basis_air_temperature"
    ]
    
    prep[invalid_zero] = prep[invalid_zero].replace(0, np.nan)
    prep[invalid_zero] = prep[invalid_zero].fillna(method='ffill', limit=12) # Revisar bé això

    prep = prep.dropna(subset=['glucose_level'])

    prep = prep.drop(columns=['time'])
    return prep

#### Proves de visualització del dataset

In [85]:
prova_559_train = preprocess(prova_559_train)
prova_559_test = preprocess(prova_559_test)

print('Shape train: ', prova_559_train.shape)
print('Shape test: ', prova_559_test.shape)
print('\n')
print(prova_559_train.isnull().mean()*100)

Shape train:  (11226, 25)
Shape test:  (2626, 24)


year                      0.000000
month                     0.000000
day                       0.000000
hour                      0.000000
minute                    0.000000
glucose_level             0.000000
basal                     0.000000
bolus                     0.000000
sleep                     0.000000
work                      0.000000
stressors                 0.000000
hypo_event                0.000000
illness                   0.000000
exercise                  0.000000
basis_heart_rate          2.030999
basis_gsr                 2.048815
basis_skin_temperature    2.048815
basis_air_temperature     2.048815
basis_step                0.000000
basis_sleep               0.000000
meal_Almuerzo             0.000000
meal_Cena                 0.000000
meal_Correccion_hipo      0.000000
meal_Desayuno             0.000000
meal_Snack                0.000000
dtype: float64


## Entrenament del model

#### Definició de la funcio de split features i target

In [86]:
def make_xy(df, pas):

    y = df['glucose_level'].shift(-pas)

    X = df.iloc[:-pas].copy() # totes les columnes
    y = y.iloc[:-pas] # mateixes files que X

    return X, y



#### Definició de la funció de evaluació del model

In [87]:
def evaluate(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    return rmse, mae

### Entrenament per cada pacient (model offline)

In [88]:
resultats = []

for pacient in PACIENTS:
    print(f'\nPacient {pacient}')
    train_raw = load_data(pacient, 'train')
    test_raw  = load_data(pacient, 'test')

    train = preprocess(train_raw)
    test  = preprocess(test_raw)

    # Igualem les columnes del train y del test i emplenem el que falti amb 0
    train, test = train.align(test, join='outer', axis=1, fill_value=0)

    # Ignorem els 60 minuts primers del test
    test = test.iloc[12:].reset_index(drop=True)

    for minuts, steps in HORITZO.items():
        # Entrenament del model
        X_train, y_train = make_xy(train, steps)

        model = RandomForestRegressor(
            n_estimators=600,
            max_depth=10,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)

        # Predicció del test
        X_test = test.iloc[:-steps].copy()
        y_true = test['glucose_level'].shift(-steps).iloc[:-steps].reset_index(drop=True)


        y_pred = model.predict(X_test)

        # guardem la predicció
        directori_pred = f'../data/predicted/pred{pacient}_{minuts}min.csv'

        df_pred = pd.DataFrame({'idx_original': X_test.index, f'pred_glucosa_t+{minuts}': y_pred})
        df_pred.to_csv(directori_pred, index=False)

        # Evaluació del model
        rmse, mae = evaluate(y_true, y_pred)
        resultats.append({
            'Pacient': pacient,
            'Horitzo': minuts,
            'RMSE': rmse,
            'MAE' : mae
        })
        print(f'{minuts} min: RMSE={rmse:.2f}  MAE={mae:.2f}')




Pacient 559
30 min: RMSE=25.05  MAE=17.47
60 min: RMSE=39.18  MAE=28.74

Pacient 563
30 min: RMSE=21.34  MAE=15.77
60 min: RMSE=34.75  MAE=25.69

Pacient 570
30 min: RMSE=18.98  MAE=13.48
60 min: RMSE=31.45  MAE=23.56

Pacient 575
30 min: RMSE=24.86  MAE=18.63
60 min: RMSE=56.06  MAE=42.74

Pacient 588
30 min: RMSE=21.83  MAE=16.03
60 min: RMSE=33.96  MAE=25.12

Pacient 591
30 min: RMSE=26.35  MAE=19.84
60 min: RMSE=41.77  MAE=33.18


## Taula final de resutats

In [89]:
resultats_df = pd.DataFrame(resultats)

taula = (resultats_df.pivot(index='Pacient', columns='Horitzo', values=['RMSE','MAE']))
print()
print(f'Visualització inicial de la taula: {taula}')

# Renombrem les columnes de la taula
taula.columns = ['RMSE 30','RMSE 60','MAE 30', 'MAE 60']

# Reordenem les columnes
taula = taula[['RMSE 30','MAE 30','RMSE 60','MAE 60']]

# Calculem el promig
promig = taula.mean().to_frame().T
promig.index = ['PROMIG'] 

# Mostrem la taula final amb el promig
taula_final = pd.concat([taula, promig], axis=0)

print("\nRESULTATS FINALS:")
print(taula_final)


Visualització inicial de la taula:               RMSE                   MAE           
Horitzo         30         60         30         60
Pacient                                            
559      25.046695  39.182130  17.473700  28.738606
563      21.341972  34.745526  15.770565  25.685682
570      18.980363  31.447164  13.479377  23.561962
575      24.862865  56.060732  18.632375  42.744356
588      21.827429  33.961035  16.028151  25.116500
591      26.347734  41.768241  19.841430  33.179282

RESULTATS FINALS:
          RMSE 30     MAE 30    RMSE 60     MAE 60
559     25.046695  17.473700  39.182130  28.738606
563     21.341972  15.770565  34.745526  25.685682
570     18.980363  13.479377  31.447164  23.561962
575     24.862865  18.632375  56.060732  42.744356
588     21.827429  16.028151  33.961035  25.116500
591     26.347734  19.841430  41.768241  33.179282
PROMIG  23.067843  16.870933  39.527471  29.837731
